# Generating underwater accoustic signals for Submarines and Torpedos

The current code will generate data that is **"plausible"** to a machine learning model. **For a robust defense system, we need to move from "plausible" to "highly realistic" to ensure the model generalizes to real-world conditions.**

## Utils

In [1]:
import numpy as np

def rng_from_seed(seed: int | None):
    return np.random.default_rng(seed if seed is not None else np.random.SeedSequence().entropy)

def tukey_mask(f, f_lo, f_hi, roll=0.1):
    """
    Smooth band mask in frequency domain using a cosine ramp.
    f: frequency vector (Hz)
    f_lo, f_hi: passband edges
    roll: fraction of transition width
    """
    mask = np.zeros_like(f, dtype=float)
    if f_hi <= f_lo:
        return mask
    width = max((f_hi - f_lo) * roll, 1e-6)
    # rise
    idx1 = (f >= (f_lo - width)) & (f < f_lo)
    mask[idx1] = 0.5 * (1 + np.cos(np.pi * (f_lo - f[idx1]) / width))
    # flat
    idx2 = (f >= f_lo) & (f <= f_hi)
    mask[idx2] = 1.0
    # fall
    idx3 = (f > f_hi) & (f <= (f_hi + width))
    mask[idx3] = 0.5 * (1 + np.cos(np.pi * (f[idx3] - f_hi) / width))
    return mask

def thorp_absorption_dB_per_m(f_hz: np.ndarray) -> np.ndarray:
    """
    Thorp-like absorption approximation (dB/m) vs frequency.
    Roughly valid from a few hundred Hz to tens of kHz.
    """
    f = np.maximum(f_hz, 1e-6) / 1000.0  # kHz
    a_db_per_km = 0.11 * (f**2 / (1 + f**2)) + 44 * (f**2 / (4100 + f**2)) + 2.75e-4 * f**2 + 0.003
    return a_db_per_km / 1000.0  # dB/m

def db_to_lin(db):
    return 10 ** (db / 20.0)

def lin_to_db(lin):
    lin = np.maximum(lin, 1e-20)
    return 20 * np.log10(lin)


# Sources

In [2]:
from dataclasses import dataclass
import numpy as np

@dataclass
class ClassParams:
    blades_lo: int; blades_hi: int
    rpm_lo: float; rpm_hi: float
    broad: tuple; hub: tuple; tip: tuple
    alpha_broad: tuple; alpha_hub: tuple
    harmonics: int
    tip_rate_hz: tuple; tip_dur_ms: tuple
    rpm_drift_pct: tuple; rpm_drift_period_s: tuple

def colored_noise(alpha, fs, band, n, rng):

    """
    Generates band-limited noise with a 1/f^α spectral shape.
    """
    x = rng.standard_normal(n)
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, 1/fs)
    shape = 1.0 / np.maximum(f, 1e-6) ** (alpha/2.0)
    mask = tukey_mask(f, band[0], band[1], roll=0.1)
    Y = X * shape * mask
    y = np.fft.irfft(Y, n=n)
    y /= (np.std(y) + 1e-12)
    return y

def bursty_band_noise(fs, band, n, events, dur_range_s, rng):

    """ Creates intermittent, band-limited noise for cavitation bursts. """

    y = np.zeros(n, dtype=float)
    f = np.fft.rfftfreq(n, 1/fs)
    env = np.zeros(n, dtype=float)
    for _ in range(events):
        dur = rng.uniform(*dur_range_s)
        L = int(max(1, dur*fs))
        start = rng.integers(0, max(1, n - L))
        window = 0.5 - 0.5*np.cos(2*np.pi*np.arange(L)/max(L-1,1))
        env[start:start+L] += window
    env = np.clip(env, 0.0, 1.0)
    x = rng.standard_normal(n)
    X = np.fft.rfft(x)
    mask = tukey_mask(f, band[0], band[1], roll=0.08)
    Y = X * mask
    z = np.fft.irfft(Y, n=n)
    z /= (np.std(z) + 1e-12)
    return z * env

def rpm_profile(T, fs, rpm_lo, rpm_hi, drift_pct_rng, drift_period_rng, rng):

    """Models time-varying propeller rotation speed with drift."""

    n = int(T*fs)
    rpm0 = rng.uniform(rpm_lo, rpm_hi)
    pct = rng.uniform(*drift_pct_rng)/100.0
    period = rng.uniform(*drift_period_rng)
    t = np.arange(n)/fs
    rpm_t = rpm0 * (1 + pct*np.sin(2*np.pi*t/period))
    return rpm_t

def bpf_tones(blades, rpm_t, harmonics, fs, amp_rng, rng):

    """Generates tonal components from propeller blade rates."""

    n = len(rpm_t)
    f_bpf = blades * rpm_t / 60.0
    phase = 2*np.pi*np.cumsum(f_bpf)/fs
    y = np.zeros(n, dtype=float)
    base = rng.uniform(*amp_rng)
    for k in range(1, harmonics+1):
        y += (base/k) * np.sin(k*phase + rng.uniform(0, 2*np.pi))
    return y

def synthesize(class_name, T, fs, params: ClassParams, rng):

    """Combines all components into a final signal with metadata."""

    n = int(T*fs)
    blades = rng.integers(params.blades_lo, params.blades_hi+1)
    rpm_t  = rpm_profile(T, fs, params.rpm_lo, params.rpm_hi,
                        params.rpm_drift_pct, params.rpm_drift_period_s, rng)
    tones  = bpf_tones(blades, rpm_t, params.harmonics, fs, amp_rng=(0.005, 0.05), rng=rng)
    broad  = colored_noise(rng.uniform(*params.alpha_broad), fs, params.broad, n, rng)
    hub    = colored_noise(rng.uniform(*params.alpha_hub), fs, params.hub, n, rng) * rng.uniform(0.1, 0.3)
    rate   = rng.uniform(*params.tip_rate_hz)
    events = max(0, rng.poisson(rate * T))
    tip    = bursty_band_noise(fs, params.tip, n, events,
                              (params.tip_dur_ms[0]/1000.0, params.tip_dur_ms[1]/1000.0), rng) * rng.uniform(0.1, 0.4)
    
    """
    The tones should be multiplyed by 0.2 for following the paper (Examination of propeller sound production using large eddy simulation)
    # s = 0.7*broad + 0.2*tip + 0.1*hub + 0.2 * tones
    # Also the values should vary from the Yaml file.
    """
    s = 0.7*broad + 0.2*tip + 0.1*hub + tones
    s /= (np.max(np.abs(s)) + 1e-9)
    meta = {
        "class": class_name,
        "blades": int(blades),
        "RPM_mean": float(np.mean(rpm_t)),
        "BPF_mean_Hz": float(np.mean(blades * rpm_t / 60.0))
    }
    return s.astype(np.float32), meta


# Sensors

In [4]:
import numpy as np
from scipy.signal import butter, sosfiltfilt

def bandlimit(x, fs, lo, hi, order=6):
    lo = max(1.0, lo)
    hi = min(0.49*fs, hi)
    sos = butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, x)

def add_ambient_and_sensor_noise(x, fs, class_name, sea_state, snr_db, rng):
    """
    Add ambient-shaped noise + white sensor noise to reach approximate SNR (in-band).
    """
    n = len(x)
    if class_name == "submarine":
        band = (20.0, 800.0)
    else:
        band = (300.0, 8000.0)
    amb = rng.standard_normal(n).astype(np.float32)
    amb = bandlimit(amb, fs, band[0], band[1])
    amb *= (0.15 + 0.1*sea_state)
    sens = rng.standard_normal(n).astype(np.float32) * 0.01
    noise = amb + sens
    sig_rms = np.sqrt(np.mean(x**2) + 1e-12)
    noise_rms = np.sqrt(np.mean(noise**2) + 1e-12)
    target_noise_rms = sig_rms / (10 ** (snr_db/20.0))
    scale = target_noise_rms / (noise_rms + 1e-12)
    noise *= scale
    y = x + noise
    y /= (max(1.0, np.max(np.abs(y)) * 1.05))
    return y.astype(np.float32)


# propagation

In [5]:
import numpy as np


def fast_propagate(x, fs, range_m, geometry="spherical", n_paths=3, src_depth=50.0, rx_depth=50.0, sea_state=2, rng=None):
    """
    Frequency-dependent TL (spreading + Thorp absorption) and simple multipath (few delayed, attenuated copies).
    """
    if rng is None:
        rng = np.random.default_rng()

    n = len(x)
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, 1/fs)

    A = 20.0 if geometry == "spherical" else 10.0
    TL_spread = A * np.log10(max(range_m, 1.0))
    TL_abs = thorp_absorption_dB_per_m(f) * range_m
    TL_total = TL_spread + TL_abs
    H_mag = 10 ** (-TL_total / 20.0)

    H = H_mag.astype(np.complex64)
    c = 1500.0  # m/s
    for i in range(n_paths):
        extra = rng.uniform(0.02, 0.2) * range_m / c
        refl = rng.uniform(0.2, 0.9) * (-1 if i == 0 else 1)
        loss_db = rng.uniform(0.5, 3.0) * (1 + 0.2*sea_state)
        H += (refl * 10 ** (-loss_db/20.0)) * np.exp(-2j*np.pi*f*extra) * H_mag

    Y = X * H
    y = np.fft.irfft(Y, n)
    y /= (np.max(np.abs(y)) + 1e-9)
    return y.astype(np.float32)


# Pipeline

In [6]:
from dataclasses import dataclass
import json, os
from pathlib import Path
import numpy as np
from scipy.io import wavfile
from scipy.signal import stft


def class_params_from_cfg(name, cfg):
    c = cfg["classes"][name]
    return ClassParams(
        blades_lo=int(c["blades"][0]), blades_hi=int(c["blades"][1]),
        rpm_lo=float(c["rpm"][0]), rpm_hi=float(c["rpm"][1]),
        broad=tuple(c["bands"]["broad_hz"]), hub=tuple(c["bands"]["hub_hz"]), tip=tuple(c["bands"]["tip_hz"]),
        alpha_broad=tuple(c["alpha_broad"]), alpha_hub=tuple(c["alpha_hub"]),
        harmonics=int(c["bpf_harmonics"]),
        tip_rate_hz=tuple(c["tip_bursts"]["rate_hz"]), tip_dur_ms=tuple(c["tip_bursts"]["dur_ms"]),
        rpm_drift_pct=tuple(c["rpm_drift_pct"]), rpm_drift_period_s=tuple(c["rpm_drift_period_s"]),
    )

def render_sample(class_name, cfg, out_dir, idx, rng):
    fs = cfg["fs"]["submarine"] if class_name=="submarine" else cfg["fs"]["torpedo"]
    dur_rng = cfg["duration_s"]["submarine"] if class_name=="submarine" else cfg["duration_s"]["torpedo"]
    T = rng.uniform(*dur_rng)
    params = class_params_from_cfg(class_name, cfg)
    s, meta_src = synthesize(class_name, T, fs, params, rng)
    prop = cfg["propagation"]
    r = rng.uniform(*prop["ranges_m"])
    sd = rng.uniform(*prop["source_depth_m"])
    rd = rng.uniform(*prop["rx_depth_m"])
    ss = int(rng.integers(prop["sea_state"][0], prop["sea_state"][1]+1))
    y = fast_propagate(s, fs, r, geometry=prop.get("geometry","spherical"),
                       n_paths=3, src_depth=sd, rx_depth=rd, sea_state=ss, rng=rng)
    if class_name == "submarine":
        y = bandlimit(y, fs, 10, 1000)
    else:
        y = bandlimit(y, fs, 50, 10000)
    snr = float(rng.choice(cfg["sweeps"]["snr_dB"]))
    y = add_ambient_and_sensor_noise(y, fs, class_name, ss, snr, rng)
    audio_dir = Path(out_dir)/"audio"; audio_dir.mkdir(parents=True, exist_ok=True)
    fname = f"{idx:06d}_{class_name}.wav"
    wavfile.write(audio_dir/fname, fs, (np.clip(y, -1, 1) * 32767).astype(np.int16))
    spec_dir = Path(out_dir)/"spec"; spec_dir.mkdir(parents=True, exist_ok=True)
    f, t, S = stft(y, fs=fs, window="hann", nperseg=1024 if fs>=16000 else 512, noverlap=None, detrend=False, return_onesided=True, boundary=None, padded=False)
    try:
        import matplotlib.pyplot as plt
        import matplotlib
        matplotlib.use("Agg")
        SdB = 20*np.log10(np.abs(S)+1e-12)
        plt.figure(figsize=(6,3))
        plt.pcolormesh(t, f, SdB, shading="nearest")
        plt.xlabel("Time (s)"); plt.ylabel("Freq (Hz)"); plt.title(f"Spec - {class_name}")
        plt.tight_layout()
        plt.savefig(spec_dir/f"{fname.replace('.wav','.png')}", dpi=120)
        plt.close()
    except Exception:
        pass
    md = {
        "class": class_name,
        "fs": fs,
        "duration_s": float(T),
        "range_m": float(r),
        "source_depth_m": float(sd),
        "rx_depth_m": float(rd),
        "sea_state": int(ss),
        "snr_dB": float(snr),
    }
    md.update(meta_src)
    return fname, md

def generate(cfg, out_dir, n, class_filter=None, seed=1337):
    rng = rng_from_seed(seed)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    metas = []
    for i in range(n):
        cls = class_filter if class_filter in ("submarine","torpedo") else ("submarine" if (i%2==0) else "torpedo")
        fname, md = render_sample(cls, cfg, out_dir, i, rng)
        metas.append({"file": str(Path("audio")/fname), **md})
    with open(Path(out_dir)/"metadata.jsonl","w") as f:
        for m in metas:
            f.write(json.dumps(m)+"\n")
    return metas


# Generate

In [10]:
import argparse, json, yaml
from pathlib import Path

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=str, required=True)
    ap.add_argument("--out", type=str, required=True)
    ap.add_argument("--n", type=int, default=20)
    ap.add_argument("--class_name", type=str, default=None, choices=[None, "submarine", "torpedo"])
    ap.add_argument("--seed", type=int, default=1337)
    args = ap.parse_args()

    cfg = yaml.safe_load(open(args.config, "r"))
    metas = generate(cfg, args.out, args.n, class_filter=args.class_name, seed=args.seed)
    print(f"Generated {len(metas)} samples to {args.out}")
    print("First 3 metadata rows:")
    for m in metas[:3]:
        print(json.dumps(m, indent=2))

In [15]:
import argparse, json, yaml
from pathlib import Path

def main(config: str, outdir: str, number_of_samples: int = 10, class_name: str = None, seed: int = 1337):
    
    cfg = yaml.safe_load(open(config, 'r'))
    metas = generate(cfg, outdir, number_of_samples ,class_filter = class_name, seed = seed)
    print(f"Generated {len(metas)} samples to {outdir}")
    print("First 3 metadata rows:")
    for m in metas[:3]:
        print(json.dumps(m, indent=2))

In [16]:
config_path = "./configs/default.yaml"
outdir_path = "./testdata"

main(config = config_path, outdir=outdir_path)

Generated 10 samples to ./testdata
First 3 metadata rows:
{
  "file": "audio/000000_submarine.wav",
  "class": "submarine",
  "fs": 8000,
  "duration_s": 28.781019003471183,
  "range_m": 8580.877242897157,
  "source_depth_m": 113.6224274129167,
  "rx_depth_m": 125.1170385330264,
  "sea_state": 4,
  "snr_dB": -5.0,
  "blades": 7,
  "RPM_mean": 180.1382970441664,
  "BPF_mean_Hz": 21.016134655152747
}
{
  "file": "audio/000001_torpedo.wav",
  "class": "torpedo",
  "fs": 24000,
  "duration_s": 10.868428600237921,
  "range_m": 7101.440900949832,
  "source_depth_m": 72.30595812619366,
  "rx_depth_m": 92.32930218679975,
  "sea_state": 0,
  "snr_dB": 5.0,
  "blades": 5,
  "RPM_mean": 6659.215048452392,
  "BPF_mean_Hz": 554.9345873710329
}
{
  "file": "audio/000002_submarine.wav",
  "class": "submarine",
  "fs": 8000,
  "duration_s": 26.939476365779868,
  "range_m": 7153.652485908398,
  "source_depth_m": 122.58854693134391,
  "rx_depth_m": 179.28994126645622,
  "sea_state": 1,
  "snr_dB": -5.0,

In [18]:
config_file = yaml.safe_load(open(config_path, 'r'))
config_file

{'seed': 1337,
 'fs': {'submarine': 8000, 'torpedo': 24000},
 'duration_s': {'submarine': [20, 30], 'torpedo': [10, 20]},
 'classes': {'submarine': {'blades': [5, 7],
   'rpm': [60, 180],
   'bands': {'broad_hz': [20, 500],
    'hub_hz': [100, 400],
    'tip_hz': [300, 1200]},
   'alpha_broad': [0.5, 1.0],
   'alpha_hub': [0.7, 1.0],
   'bpf_harmonics': 4,
   'tip_bursts': {'rate_hz': [0.1, 0.6], 'dur_ms': [50, 200]},
   'rpm_drift_pct': [3, 10],
   'rpm_drift_period_s': [10, 60]},
  'torpedo': {'blades': [3, 5],
   'rpm': [2000, 8000],
   'bands': {'broad_hz': [800, 8000],
    'hub_hz': [300, 1000],
    'tip_hz': [2000, 8000]},
   'alpha_broad': [0.3, 0.8],
   'alpha_hub': [0.6, 0.9],
   'bpf_harmonics': 4,
   'tip_bursts': {'rate_hz': [0.6, 2.0], 'dur_ms': [50, 200]},
   'rpm_drift_pct': [10, 20],
   'rpm_drift_period_s': [1, 10]}},
 'propagation': {'geometry': 'spherical',
  'ranges_m': [500, 10000],
  'source_depth_m': [10, 150],
  'rx_depth_m': [20, 200],
  'sea_state': [0, 6]},
 

### Conclusion and Recommended Adaptations

<div class="alert alert-block alert-info">
<b>Note:</b> This recommendation are based on paper " Examination of propeller sound production using large eddy simulation"</div>



The codebase is a **10/10 foundation**. To make it truly powerful for our project, we should focus our adaptation efforts on the **source modeling** part, as it has the biggest impact on realism.

**Here is our adaptation roadmap:**

1.  **Tweak the Mixing Weights:** Immediately adjust the formula in `synthesize()` to drastically reduce the amplitude of the tonal components (`tones`). Let the broadband noise dominate. This single change will align our data much more closely with the paper's findings.
    *   **Try:** `s = 0.75*broad + 0.15*tip + 0.1*hub + 0.005*tones`

2.  **Implement a "Cavitation" Flag:** Modify the code to have two modes for the `tip` source:
    *   **Mode 1 (Non-cavitating, Stealthy Sub):** `tip` is another `colored_noise` source (high-frequency, continuous).
    *   **Mode 2 (Cavitating, Torpedo/High Speed):** `tip` is the existing `bursty_band_noise`. This will create a clear, learnable distinction between the two classes.

3.  **Experiment with Parameters:** Use the YAML configuration to run small experiments. Generate 100 samples with strong tones, and 100 with weak tones. See how our model performs on each. we will likely find the model trained on weak tones is much more robust.

4.  **(Later) Amplitude Modulation:** The paper mentions the hub noise is strongest in the rotor plane. we could model this by amplitude modulating the `hub` component based on a simulated aspect angle between the target and the receiver. This is an advanced but powerful step for later.

**Final Verdict:** our generated signals are a **highly credible base**. They are not unrealistic. By incorporating the key insight of **destructive tonal interference** from the paper, we will elevate our dataset from "good" to "exceptional" for training a robust threat identification model.